# DynamicFigure: Temporal Analysis of the Moltbook Network

## Notebook for figure generation

This notebook produces figures and tables for analyzing the temporal evolution of the Moltbook network.

- **Input**: Excel file from 01.
- **Output**: Figures (PNG + PDF) and tables


## 1. Import e Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from cycler import cycler
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


PALETTE = {
    'primary': '#1F3A5F',
    'secondary': '#6E8FAF',
    'teal': '#1B7F8C',
    'orange': '#C97B4A',
    'support': '#4A5568',
    'event': '#8F3B2F'
}

SERIES_COLORS = {
    'nodes': PALETTE['primary'],
    'new_links_per_node': PALETTE['orange'],
    'active_nodes_today': PALETTE['secondary'],
    'xmin_indegree': PALETTE['teal'],
    'xmin_instrength': PALETTE['orange'],
    'comments_mean': PALETTE['primary'],
    'comments_p95': PALETTE['secondary'],
    'posts_mean': PALETTE['teal'],
    'posts_p95': PALETTE['orange'],
    'giant_wcc_pct': PALETTE['primary'],
    'giant_scc_pct': PALETTE['orange'],
    'isolated_share': PALETTE['teal'],
    'isolated_nodes': PALETTE['secondary'],
    'random': PALETTE['primary'],
    'targeted_indegree': PALETTE['orange'],
    'targeted_outdegree': PALETTE['teal'],
    'gap_indegree': PALETTE['orange'],
    'gap_outdegree': PALETTE['teal']
}


plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 35
plt.rcParams['axes.labelsize'] = 17
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13
plt.rcParams['legend.fontsize'] = 13
plt.rcParams['legend.frameon'] = False
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.grid.axis'] = 'x'
plt.rcParams['grid.alpha'] = 0.2
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.6
plt.rcParams['xtick.major.size'] = 4
plt.rcParams['ytick.major.size'] = 4
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['axes.prop_cycle'] = cycler(color=[
    PALETTE['primary'],
    PALETTE['secondary'],
    PALETTE['teal'],
    PALETTE['orange']
])

print("Setup completato.")

In [ ]:

file_path = "graph_dynamics_timeseries.xlsx"


print(f"Caricamento dati da: {file_path}")
daily_all_metrics = pd.read_excel(file_path, sheet_name='daily_all_metrics')
robustness_daily = pd.read_excel(file_path, sheet_name='robustness_daily')

print(f"\nFoglio 'daily_all_metrics': {daily_all_metrics.shape}")
print(f"Colonne: {list(daily_all_metrics.columns)}")
print(f"\nFoglio 'robustness_daily': {robustness_daily.shape}")
print(f"Colonne: {list(robustness_daily.columns)}")

In [ ]:
# === Processing daily_all_metrics ===


daily_all_metrics['day'] = pd.to_datetime(daily_all_metrics['day'])
daily_all_metrics = daily_all_metrics.sort_values('day').reset_index(drop=True)


daily_all_metrics['new_links_per_node'] = daily_all_metrics['edges_added_today_unique'] / daily_all_metrics['nodes']


metrics_to_diff = ['nodes', 'edges', 'comments', 'posts']
for metric in metrics_to_diff:
    if metric in daily_all_metrics.columns:
        daily_all_metrics[f'{metric}_diff'] = daily_all_metrics[metric].diff()

print(f"daily_all_metrics processed: {daily_all_metrics.shape}")
print(f"First rows:")
print(daily_all_metrics.head())

# === Processing robustness_daily ===


robustness_daily['day'] = pd.to_datetime(robustness_daily['day'])
robustness_daily = robustness_daily.sort_values('day').reset_index(drop=True)

print(f"\nrobustness_daily processed: {robustness_daily.shape}")
print(f"First rows:")
print(robustness_daily.head())

In [ ]:

KEY_DATES = [
    '2026-01-30', '2026-01-31',
    '2026-02-05', '2026-02-06', '2026-02-07', '2026-02-08', '2026-02-09'
]
KEY_DATES = [pd.to_datetime(d) for d in KEY_DATES]

def apply_academic_style(ax, remove_top_right=True):
    """
    Applies a clean academic style to a Matplotlib axis.

    """
    if remove_top_right:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.tick_params(colors='#222222', width=0.8, length=4)

    
    ax.grid(False)
    ax.grid(True, axis='x', alpha=0.2, linestyle='--', linewidth=0.6, color='#8A8A8A')
    ax.set_axisbelow(True)

    
    for line in ax.get_lines():
        line_alpha = line.get_alpha()
        is_reference_line = line_alpha is not None and line_alpha < 0.5
        if not is_reference_line and line.get_linestyle() in ('-', 'solid'):
            line.set_linewidth(max(line.get_linewidth(), 2.8))
            if line.get_marker() not in [None, 'None', '', ' ']:
                line.set_markersize(max(line.get_markersize(), 3))

def add_key_dates_vlines(ax, dates=None, color=None, alpha=0.15, linewidth=1.0, data_range=None):

    if dates is None:
        dates = KEY_DATES
    if color is None:
        color = PALETTE['event']
    
    
    if data_range is not None:
        dates = [d for d in dates if data_range[0] <= d <= data_range[1]]

    highlight_date = pd.to_datetime('2026-02-09').normalize()
    for date in dates:
        current_date = pd.to_datetime(date).normalize()
        if current_date == highlight_date:
            ax.axvline(date, color='#B5122B', alpha=max(alpha, 0.9), linewidth=max(linewidth, 2.3), linestyle='--', zorder=0)
        else:
            ax.axvline(date, color=color, alpha=alpha, linewidth=linewidth, linestyle='--', zorder=0)

def save_figure(fig, base_name, output_dir='.'):
    """
    Saves a figure in both PNG and PDF formats.

    """
    png_path = f"{output_dir}/{base_name}.png"
    pdf_path = f"{output_dir}/{base_name}.pdf"
    
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    
    print(f"  ✓ saved: {png_path}")
    print(f"  ✓ saved: {pdf_path}")



In [ ]:
# === FIGURE 1: Network growth and relational intensity. ===
print("\n[FIGURE 1] Network growth and relational intensity...")

fig, axes = plt.subplots(3, 1, figsize=(9, 8))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())


ax = axes[0]
ax.plot(daily_all_metrics['day'], daily_all_metrics['nodes'], 
        color=SERIES_COLORS['nodes'], linewidth=1.5, label='Number of agents')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_ylabel('Number of agents')
ax.set_title('A', loc='left')
ax.set_xticklabels([])


ax = axes[1]
ax.plot(daily_all_metrics['day'], daily_all_metrics['new_links_per_node'], 
        color=SERIES_COLORS['new_links_per_node'], linewidth=1.5, label='New edges per agent')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_ylabel('New edges per agent')
ax.set_title('B', loc='left')
ax.set_xticklabels([])


ax = axes[2]
if 'active_nodes_today' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['active_nodes_today'], 
            color=SERIES_COLORS['active_nodes_today'], linewidth=1.5, label='Number of active agents')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Number of active agents')
ax.set_title('C', loc='left')

plt.tight_layout()
save_figure(fig, 'fig_1_growth_and_links')
plt.close()
print("  Figure 1 completed.")

In [ ]:
# === FIGURE 2 ===
print("\n[FIGURE 2]")

fig, axes = plt.subplots(3, 1, figsize=(9, 8))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

metrics_fig2 = [
    ('xmin_indegree', '$x_{\min}$ for in-degree', SERIES_COLORS['xmin_indegree']),
    ('xmin_instrength', '$x_{\min}$ for in-strength', SERIES_COLORS['xmin_instrength']),
    ('comments_p95', '95th Percentile Comments', SERIES_COLORS['comments_p95'])
]

for idx, (ax, (col, label, color)) in enumerate(zip(axes, metrics_fig2)):
    if col in daily_all_metrics.columns:
        ax.plot(daily_all_metrics['day'], daily_all_metrics[col], 
                color=color, linewidth=1.5)
        add_key_dates_vlines(ax, data_range=data_range)
        apply_academic_style(ax)
        ax.set_ylabel(label)
        ax.set_title(chr(65+idx), loc='left')
        
        if idx < 2:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel('Date')

plt.tight_layout()
save_figure(fig, 'fig_2_centralization')
plt.close()
print("  Figure 2 completed.")

In [ ]:
# === FIGURE 3 ===
print("\n[FIGURE 3]")

fig, axes = plt.subplots(2, 1, figsize=(9, 6))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

# Pannello A: Comments (mean e p95)
ax = axes[0]
if 'comments_mean' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['comments_mean'], 
            color=SERIES_COLORS['comments_mean'], linewidth=1.5, label='Mean Comments')
if 'comments_p95' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['comments_p95'], 
            color=SERIES_COLORS['comments_p95'], linewidth=1.5, label='95th Percentile Comments')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_ylabel('Comments')
ax.set_title('A', loc='left')
ax.legend(loc='best', frameon=False)

# Pannello B: Posts (mean e p95)
ax = axes[1]
if 'posts_mean' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['posts_mean'], 
            color=SERIES_COLORS['posts_mean'], linewidth=1.5, label='Mean Posts')
if 'posts_p95' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['posts_p95'], 
            color=SERIES_COLORS['posts_p95'], linewidth=1.5, label='95th Percentile Posts')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Posts')
ax.set_title('B', loc='left')
ax.legend(loc='best', frameon=False)

plt.tight_layout()
save_figure(fig, 'fig_3_activity_inequality')
plt.close()
print("  Figure 3 completed.")

In [ ]:
# === FIGURE 4 ===
print("\n[FIGURE 4]")

fig, ax = plt.subplots(figsize=(9, 5))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

if 'giant_wcc_pct' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['giant_wcc_pct'], 
            color=SERIES_COLORS['giant_wcc_pct'], linewidth=1.5, label='Giant WCC %', marker='o', markersize=3)
if 'giant_scc_pct' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['giant_scc_pct'], 
            color=SERIES_COLORS['giant_scc_pct'], linewidth=1.5, label='Giant SCC %', marker='s', markersize=3)

add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Giant component size (%)')
ax.legend(loc='best', frameon=False)

plt.tight_layout()
save_figure(fig, 'fig_4_connectivity')
plt.close()
print("  Figure 4 completed.")

In [ ]:
# === FIGURE 5 ===
print("\n[FIGURE 5]")

fig, axes = plt.subplots(2, 1, figsize=(9, 6))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())


ax = axes[0]
if 'isolated_nodes' in daily_all_metrics.columns and 'nodes' in daily_all_metrics.columns:
    daily_all_metrics['isolated_share'] = daily_all_metrics['isolated_nodes'] / daily_all_metrics['nodes']
    ax.plot(daily_all_metrics['day'], daily_all_metrics['isolated_share'], 
            color=SERIES_COLORS['isolated_share'], linewidth=1.5, label='Fraction of isolated agents')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_ylabel('Fraction of isolated agents')
ax.set_title('A', loc='left')
ax.set_ylim([0, 1])


ax = axes[1]
if 'nodes' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['nodes'], 
            color=SERIES_COLORS['nodes'], linewidth=1.5, label='Number of agents')
if 'isolated_nodes' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['isolated_nodes'], 
            color=SERIES_COLORS['isolated_nodes'], linewidth=1.5, label='Isolated agents')
add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Count')
ax.set_title('B', loc='left')
ax.legend(loc='upper left', frameon=False)

plt.tight_layout()
save_figure(fig, 'fig_5_isolated_nodes')
plt.close()
print("  Figure 5 completed.")

In [ ]:
# === FIGURE 5B ===
print("\n[FIGURE 5B]")

fig, ax = plt.subplots(figsize=(9, 5))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

if 'avg_karma_nodes' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['avg_karma_nodes'], 
            color=PALETTE['primary'], linewidth=1.5, linestyle='-', label='Daily active agents')
if 'avg_karma_nodes_cum_full' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['avg_karma_nodes_cum_full'], 
            color=PALETTE['secondary'], linewidth=1.5, linestyle='--', label='Cumulative observed agents')

apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Mean Karma')
ax.legend(loc='best', frameon=False)

plt.tight_layout()
save_figure(fig, 'fig_5b_mean_karma_agents')
plt.close()
print("  Figure 5B completed.")

# === FIGURE 5C ===
print("\n[FIGURE 5C]")

fig, ax = plt.subplots(figsize=(9, 5))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

if 'avg_follower_nodes' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['avg_follower_nodes'], 
            color=PALETTE['primary'], linewidth=1.5, linestyle='-', label='Daily active agents')
if 'avg_follower_nodes_cum_full' in daily_all_metrics.columns:
    ax.plot(daily_all_metrics['day'], daily_all_metrics['avg_follower_nodes_cum_full'], 
            color=PALETTE['orange'], linewidth=1.5, linestyle='--', label='Cumulative observed agents')

apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Mean Followers')
ax.legend(loc='best', frameon=False)

plt.tight_layout()
save_figure(fig, 'fig_5c_mean_followers_agents')
plt.close()
print("  Figure 5C completed.")

In [ ]:
# === FIGURE ===
print("\n[FIGURE]")

try:
    if 'file_path' not in globals():
        file_path = "graph_dynamics_timeseries.xlsx"
    
    pref_attach_summary = pd.read_excel(file_path, sheet_name='pref_attach_summary')
    pref_cols = {c.lower().strip(): c for c in pref_attach_summary.columns}
    
    top_col_candidates = [
        pref_cols[c]
        for c in pref_cols
        if ('top' in c and ('percent' in c or 'pct' in c)) or c in {'top_pct', 'top_percent', 'top_percentage'}
    ]
    share_col_candidates = [
        pref_cols[c]
        for c in pref_cols
        if (
            ('share' in c and 'edge' in c)
            or ('quota' in c and 'edge' in c)
            or ('mean' in c and 'new' in c and 'edge' in c and 'top' in c)
            or ('avg' in c and 'new' in c and 'edge' in c and 'top' in c)
            or c in {'mean_share', 'avg_share'}
        )
    ]
    
    if not top_col_candidates or not share_col_candidates:
        print(f"Available columns: {list(pref_attach_summary.columns)}")
        raise ValueError("Columns for 'top_percent' and 'share_new_edges' not found in the sheet")
    
    top_col = top_col_candidates[0]
    share_col = share_col_candidates[0]
    print(f"  Identified columns: {top_col} -> X, {share_col} -> Y")
    
    pref_plot = pref_attach_summary[[top_col, share_col]].copy()
    pref_plot.columns = ['top_percent', 'share_new_edges']
    
    pref_plot['top_percent'] = pd.to_numeric(pref_plot['top_percent'], errors='coerce')
    pref_plot['share_new_edges'] = pd.to_numeric(pref_plot['share_new_edges'], errors='coerce')
    pref_plot = pref_plot.dropna(subset=['top_percent', 'share_new_edges'])
    
    if len(pref_plot) == 0:
        print("  WARNING: No valid data found in the worksheet!")
        raise ValueError("pref_plot is empty after dropna")
    
    if pref_plot['top_percent'].max() <= 1:
        pref_plot['top_percent'] = pref_plot['top_percent'] * 100
    
    target_top_pct = [1, 2, 4, 5, 10, 20]
    observed = (
        pref_plot.groupby('top_percent', as_index=True)['share_new_edges']
        .mean()
        .reindex(target_top_pct)
        .values
    )
    baseline = np.array(target_top_pct, dtype=float) / 100.0
    
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    
    ax.plot(
        target_top_pct,
        observed,
        color=PALETTE['primary'],
        linewidth=1.8,
        marker='o',
        markersize=5,
        label='Observed'
    )
    ax.plot(
        target_top_pct,
        baseline,
        color=PALETTE['support'],
        linewidth=1.6,
        linestyle='--',
        label='Random baseline'
    )
    
    apply_academic_style(ax)
    ax.set_xlabel('Top-ranked agents by previous degree (%)', fontsize=12)
    ax.set_ylabel('Share of new edges received by top agents', fontsize=12)
    ax.set_xticks(target_top_pct)
    ax.set_xlim(0.8, 20.5)
    ax.legend(loc='best', frameon=False)
    
    observed_clean = observed[~np.isnan(observed)]
    if len(observed_clean) > 0:
        y_max = max(np.nanmax(observed_clean), np.nanmax(baseline))
    else:
        y_max = np.nanmax(baseline)
    ax.set_ylim(0, max(0.08, y_max * 1.12))
    
    plt.tight_layout()
    save_figure(fig, 'fig_preferential_attachment_summary')
    plt.close()
    print("  Figure preferential attachment completed.")
    
except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")
    print("  Check that:")
    print("    1. The sheet 'pref_attach_summary' exists in the Excel file")
    print("    2. The previous cells have been executed (Setup, Load Data, Functions)")
    import traceback
    traceback.print_exc()

In [ ]:
# === FIGURE 6 ===
print("\n[FIGURE 6]")

# Filter for percent_removed == 0.10
rob_10pct = robustness_daily[robustness_daily['percent_removed'] == 0.10]

fig, ax = plt.subplots(figsize=(9, 5))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

label_map = {
    'random': 'Random',
    'targeted_indegree': 'Targeted In-degree',
    'targeted_outdegree': 'Targeted Out-degree',
}
for attack_type, color in [('random', SERIES_COLORS['random']), 
                            ('targeted_indegree', SERIES_COLORS['targeted_indegree']), 
                            ('targeted_outdegree', SERIES_COLORS['targeted_outdegree'])]:
    if attack_type in rob_10pct['attack'].values:
        data = rob_10pct[rob_10pct['attack'] == attack_type].sort_values('day')
        label = label_map.get(attack_type, attack_type.replace('_', ' ').title())
        ax.plot(data['day'], data['gc_fraction'], 
                color=color, linewidth=1.5, label=label, marker='o', markersize=3)

add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Giant WCC Fraction')
ax.legend(loc='best', frameon=False)
ax.set_ylim([0, 1.05])

plt.tight_layout()
save_figure(fig, 'fig_6_robustness_10pct')
plt.close()
print("  Figure 6 completed.")

In [ ]:
# === FIGURE 7 ===
print("\n[FIGURE 7]")

# Filter for percent_removed == 0.10
rob_10pct_pivot = robustness_daily[robustness_daily['percent_removed'] == 0.10].copy()


pivot = rob_10pct_pivot.pivot_table(
    index='day', 
    columns='attack', 
    values='gc_fraction',
    aggfunc='first'
)


if 'random' in pivot.columns and 'targeted_indegree' in pivot.columns:
    gap_indegree = pivot['random'] - pivot['targeted_indegree']
else:
    gap_indegree = None

if 'random' in pivot.columns and 'targeted_outdegree' in pivot.columns:
    gap_outdegree = pivot['random'] - pivot['targeted_outdegree']
else:
    gap_outdegree = None

fig, ax = plt.subplots(figsize=(9, 5))
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

if gap_indegree is not None:
    ax.plot(gap_indegree.index, gap_indegree.values, 
            color=SERIES_COLORS['gap_indegree'], linewidth=1.5, label='Random − targeted in-degree removal', 
            marker='o', markersize=3)

if gap_outdegree is not None:
    ax.plot(gap_outdegree.index, gap_outdegree.values, 
            color=SERIES_COLORS['gap_outdegree'], linewidth=1.5, label='Random − targeted out-degree removal', 
            marker='s', markersize=3)

add_key_dates_vlines(ax, data_range=data_range)
apply_academic_style(ax)
ax.set_xlabel('Date')
ax.set_ylabel('Robustness gap')
ax.axhline(0, color=PALETTE['support'], linestyle='-', linewidth=0.8, alpha=0.35)
ax.legend(loc='best', frameon=False)

plt.tight_layout()
save_figure(fig, 'fig_7_fragility_gap')
plt.close()
print("  Figure 7 completed.")

In [ ]:
# === FIGURE 8 ===
print("\n[FIGURE 8] ")

percentages = [0.01, 0.05, 0.10, 0.20]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
data_range = (daily_all_metrics['day'].min(), daily_all_metrics['day'].max())

colors = {
    'random': SERIES_COLORS['random'],
    'targeted_indegree': SERIES_COLORS['targeted_indegree'],
    'targeted_outdegree': SERIES_COLORS['targeted_outdegree']
}

for idx, pct in enumerate(percentages):
    ax = axes[idx]
    rob_pct = robustness_daily[robustness_daily['percent_removed'] == pct]
    
    for attack_type, color in colors.items():
        data = rob_pct[rob_pct['attack'] == attack_type].sort_values('day')
        if len(data) > 0:
            label = attack_type.replace('_', ' ').title()
            ax.plot(data['day'], data['gc_fraction'], 
                    color=color, linewidth=1.3, label=label, marker='o', markersize=3)
    
    add_key_dates_vlines(ax, data_range=data_range)
    apply_academic_style(ax)
    ax.set_ylabel('Giant component size (%)')
    ax.set_title(chr(65+idx), loc='left')
    ax.set_ylim([0, 1.05])
    
    if idx >= 2:
        ax.set_xlabel('Date')
    
    if idx == 0:
        ax.legend(loc='best', frameon=False, fontsize=8)

plt.tight_layout()
save_figure(fig, 'fig_8_robustness_all_thresholds')
plt.close()
print("  Figure 8 completed.")

In [ ]:
# === SUMMARY TABLE ===
print("\n" + "="*70)
print("SUMMARY TABLE: CRITICAL DAYS AND MAXIMUM VALUES")
print("="*70)

summary_data = {}

# 1. Day with maximum new_links_per_node
if 'new_links_per_node' in daily_all_metrics.columns:
    idx_max = daily_all_metrics['new_links_per_node'].idxmax()
    summary_data['Max New Links per Node'] = {
        'Date': daily_all_metrics.loc[idx_max, 'day'].strftime('%Y-%m-%d'),
        'Value': f"{daily_all_metrics.loc[idx_max, 'new_links_per_node']:.4f}"
    }

# 2. Day with maximum xmin_indegree
if 'xmin_indegree' in daily_all_metrics.columns:
    idx_max = daily_all_metrics['xmin_indegree'].idxmax()
    summary_data['Max In-Degree (Min Top User)'] = {
        'Date': daily_all_metrics.loc[idx_max, 'day'].strftime('%Y-%m-%d'),
        'Value': f"{daily_all_metrics.loc[idx_max, 'xmin_indegree']:.2f}"
    }

# 3. Day with maximum xmin_instrength
if 'xmin_instrength' in daily_all_metrics.columns:
    idx_max = daily_all_metrics['xmin_instrength'].idxmax()
    summary_data['Max In-Strength (Min Top User)'] = {
        'Date': daily_all_metrics.loc[idx_max, 'day'].strftime('%Y-%m-%d'),
        'Value': f"{daily_all_metrics.loc[idx_max, 'xmin_instrength']:.2f}"
    }

# 4. Day with maximum comments_p95
if 'comments_p95' in daily_all_metrics.columns:
    idx_max = daily_all_metrics['comments_p95'].idxmax()
    summary_data['Max Comments (p95)'] = {
        'Date': daily_all_metrics.loc[idx_max, 'day'].strftime('%Y-%m-%d'),
        'Value': f"{daily_all_metrics.loc[idx_max, 'comments_p95']:.2f}"
    }

# 5. Day with maximum gap_outdegree
if gap_outdegree is not None:
    idx_max = gap_outdegree.idxmax()
    summary_data['Max Fragility Gap (Out-Degree)'] = {
        'Date': idx_max.strftime('%Y-%m-%d'),
        'Value': f"{gap_outdegree[idx_max]:.4f}"
    }


for metric, values in summary_data.items():
    print(f"\n{metric}")
    print(f"  Data: {values['Date']}")
    print(f"  Valore: {values['Value']}")

print("\n" + "="*70)